# VeriCall — Rapid Smoke Test & Pipeline Verification (2-Minute Run)
### Low-Compute Dry Run on Colab (T4 / L4 / CPU / Any GPU)
**Author:** Bhuvnesh Sahu ([@bhuvneshsahu01](https://github.com/bhuvneshsahu01))

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhuvneshsahu01/vericall/blob/main/VeriCall_Fast_Smoke_Test.ipynb)

---
### Why This Notebook Exists
Before launching the full post-training run on 2,500 samples, this notebook runs the **exact same 10-stage pipeline** using a **1% miniature slice**:
- **Dataset Slice:** 50 SFT training samples & 25 evaluation requests.
- **DPO Pairs:** 25 execution-verified near-miss pairs.
- **Model:** `Qwen/Qwen2.5-1.5B-Instruct` (ultra-fast, loads in seconds, uses <2 GB VRAM) or `Qwen2.5-7B-Instruct`.
- **Total Runtime:** **~2 to 3 minutes** on free Colab T4.
- **Outcome:** Validates every single component, import, sandbox endpoint, DPO miner, training loop, and evaluation renderer!


### Stage 1: Environment Setup & Dependencies


In [ ]:
# Install dependencies
!pip install --quiet --upgrade pip
!pip install --quiet transformers trl peft accelerate bitsandbytes
!pip install --quiet fastapi uvicorn httpx pydantic statsmodels scipy scikit-learn tabulate

import os
import sys
from pathlib import Path

# Verify and set working directory to repository root
if os.path.exists("/content"):
    if not os.path.exists("/content/vericall"):
        os.system("git clone https://github.com/bhuvneshsahu01/vericall.git /content/vericall")
    os.chdir("/content/vericall")
else:
    curr = Path.cwd()
    if (curr / "pyproject.toml").exists():
        print(f"[✓] Already at repository root: {curr}")
    elif (curr.parent / "pyproject.toml").exists():
        os.chdir(str(curr.parent))
    else:
        print(f"[!] Current directory: {curr}")

sys.path.insert(0, os.getcwd())
print("[✓] Environment ready. Working directory:", os.getcwd())


### Stage 2: FastAPI Mock Sandbox Verification


In [ ]:
from vericall.sandbox.validator import ToolCallValidator
from vericall.sandbox.app import app

validator = ToolCallValidator()
print(f"[✓] Initialized ToolCallValidator with {len(validator.registry)} endpoints.")

# Quick smoke test of 4 endpoints across 4 domains
smoke_cases = [
    ("get_stock_quote", {"symbol": "NVDA"}),
    ("search_products", {"query": "headphones", "category": "electronics"}),
    ("provision_server", {"instance_type": "c5.xlarge", "region": "us-east-1", "count": 1, "environment": "dev"}),
    ("search_flights", {"origin_airport": "JFK", "destination_airport": "SFO", "departure_date": "2026-11-15", "passengers": 1, "cabin_class": "economy"}),
]

for fn, args in smoke_cases:
    res = validator.validate_call(fn, args)
    assert res.is_success, f"Validation failed for {fn}: {res.error_message}"

print("[✓] Sandbox endpoints verified functional.")


### Stage 3: Load Fast Model (Qwen2.5-1.5B or 7B)


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Use Qwen2.5-1.5B for lightning-fast smoke test (~1.5GB VRAM, loads in 10s)
# Switch to 'Qwen/Qwen2.5-7B-Instruct' whenever you want the full model!
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

print(f"[*] Loading {MODEL_ID} for rapid smoke testing...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4-bit quantization if GPU available
if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
else:
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32, trust_remote_code=True)

device = next(model.parameters()).device
vram_gb = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0.0
print(f"[✓] Model loaded on {device}. VRAM allocated: {vram_gb:.2f} GB")


### Stage 4: Base Model Baseline Evaluation (Mini-Slice: 25 Requests)


In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
from vericall.eval.evaluate import evaluate_predictions

unseen_path = Path("data/eval_unseen_apis.jsonl")
all_records = [json.loads(line) for line in open(unseen_path, "r", encoding="utf-8") if line.strip()]

# Take a fast 25-sample slice for testing
SMOKE_EVAL_SIZE = 25
test_records = all_records[:SMOKE_EVAL_SIZE]
print(f"[*] Running baseline evaluation on {len(test_records)} requests...")

TOOL_SYSTEM_PROMPT = """You are a precise, tool-calling assistant. Given the user's request, produce a JSON object representing the exact tool call with the required parameters.

Output format:
{"name": "<function_name>", "arguments": {<key_value_pairs>}}
"""

model.eval()
base_predictions = []
for item in tqdm(test_records):
    prompt = f"<|im_start|>system\n{TOOL_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['query']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=96, use_cache=True, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(response_text)
        fn = parsed.get("name", "")
        args = parsed.get("arguments", {})
    except Exception:
        fn = "invalid_json"
        args = {}

    base_predictions.append({"name": fn, "arguments": args})

base_results = evaluate_predictions(test_records, base_predictions, "base_model")
print(f"\n[Base Model] Smoke Accuracy: {base_results['accuracy_pct']}% ({base_results['passed_requests']}/{base_results['total_requests']})")
print(f"             Errors: {base_results['error_taxonomy']}")


### Stage 5: QLoRA SFT Training (Mini-Slice: 50 Samples, 10 Steps)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from vericall.train.train_sft import load_sft_dataset

if torch.cuda.is_available():
    model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, peft_config)

# Slice 50 samples for rapid 1-minute training
sft_data = load_sft_dataset()[:50]
train_dataset = Dataset.from_list(sft_data)

training_args = TrainingArguments(
    output_dir="checkpoints/smoke_sft",
    per_device_train_batch_size=2,
    max_steps=10,  # 10 quick steps
    learning_rate=2e-4,
    fp16=torch.cuda.is_available(),
    logging_steps=2,
    optim="adamw_torch",
    seed=42,
)

import inspect
sft_kwargs = {
    "model": model,
    "train_dataset": train_dataset,
    "dataset_text_field": "text",
    "args": training_args,
}
sig = inspect.signature(SFTTrainer.__init__).parameters
if "processing_class" in sig:
    sft_kwargs["processing_class"] = tokenizer
else:
    sft_kwargs["tokenizer"] = tokenizer
if "max_seq_length" in sig:
    sft_kwargs["max_seq_length"] = 1024

trainer = SFTTrainer(**sft_kwargs)

print("[*] Starting fast SFT dry run (10 steps)...")
trainer.train()
print("[✓] SFT dry run complete!")



### Stage 6: SFT Model Evaluation (25 Requests)


In [ ]:
model.eval()
sft_predictions = []
print("[*] Evaluating SFT Model...")
for item in tqdm(test_records):
    prompt = f"<|im_start|>system\n{TOOL_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['query']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=96, use_cache=True, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(response_text)
        fn = parsed.get("name", "")
        args = parsed.get("arguments", {})
    except Exception:
        fn = "invalid_json"
        args = {}

    sft_predictions.append({"name": fn, "arguments": args})

sft_results = evaluate_predictions(test_records, sft_predictions, "sft_only")
print(f"\n[SFT Model] Smoke Accuracy: {sft_results['accuracy_pct']}% ({sft_results['passed_requests']}/{sft_results['total_requests']})")
print(f"            Errors: {sft_results['error_taxonomy']}")


### Stage 7: DPO Near-Miss Mining (Mini-Slice: 25 Pairs)


In [ ]:
from vericall.data.mine_dpo import mine_dpo_pairs

# Mine 25 verified pairs in seconds
dpo_pairs = mine_dpo_pairs(num_pairs=25)
print(f"[✓] Mined {len(dpo_pairs)} execution-verified preference pairs.")
print("Sample Mined Pair:")
print("  Chosen:  ", dpo_pairs[0]["chosen"])
print("  Rejected:", dpo_pairs[0]["rejected"], f"({dpo_pairs[0]['rejected_status']})")


### Stage 8: QLoRA DPO Alignment (5 Steps)


In [ ]:
from trl import DPOTrainer, DPOConfig
from vericall.train.train_dpo import load_dpo_dataset

dpo_config = DPOConfig(
    output_dir="checkpoints/smoke_dpo",
    beta=0.1,
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    max_steps=5,  # 5 fast steps
    optim="adamw_torch",
    logging_steps=1,
    seed=42,
    max_length=1024,
    max_prompt_length=512,
    fp16=torch.cuda.is_available(),
)

dpo_pairs_data = load_dpo_dataset()[:25]
dpo_train_dataset = Dataset.from_list(dpo_pairs_data)

import inspect
dpo_kwargs = {
    "model": model,
    "ref_model": None,
    "args": dpo_config,
    "train_dataset": dpo_train_dataset,
}
sig_dpo = inspect.signature(DPOTrainer.__init__).parameters
if "processing_class" in sig_dpo:
    dpo_kwargs["processing_class"] = tokenizer
else:
    dpo_kwargs["tokenizer"] = tokenizer

dpo_trainer = DPOTrainer(**dpo_kwargs)

print("[*] Starting fast DPO alignment dry run (5 steps)...")
dpo_trainer.train()
print("[✓] DPO dry run complete!")



### Stage 9: Final Statistical Evaluation & McNemar Tests


In [ ]:
from vericall.eval.stats import compute_mcnemar_test

model.eval()
dpo_predictions = []
print("[*] Generating final predictions...")
for item in tqdm(test_records):
    prompt = f"<|im_start|>system\n{TOOL_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['query']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=96, use_cache=True, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(response_text)
        fn = parsed.get("name", "")
        args = parsed.get("arguments", {})
    except Exception:
        fn = "invalid_json"
        args = {}

    dpo_predictions.append({"name": fn, "arguments": args})

dpo_results = evaluate_predictions(test_records, dpo_predictions, "sft_dpo_vericall")

stat_base, p_base = compute_mcnemar_test(base_results["binary_scores"], dpo_results["binary_scores"])
stat_sft, p_sft = compute_mcnemar_test(sft_results["binary_scores"], dpo_results["binary_scores"])

print(f"\n=======================================================")
print(f"SMOKE TEST BENCHMARK RESULTS ({SMOKE_EVAL_SIZE} Requests)")
print(f"=======================================================")
print(f"[Base Model]     Accuracy: {base_results['accuracy_pct']}%  95% CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"[SFT-Only]       Accuracy: {sft_results['accuracy_pct']}%  95% CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"[VeriCall (DPO)] Accuracy: {dpo_results['accuracy_pct']}%  95% CI: [{dpo_results['bootstrap_95_ci'][0]}%, {dpo_results['bootstrap_95_ci'][1]}%]")
print(f"McNemar Test vs Base: chi2 = {stat_base:.4f}, p = {p_base:.4e}")
print(f"=======================================================")


### Stage 10: Save Summary & Dynamic Results Rendering


In [ ]:
from pathlib import Path
from scripts.render_results import format_markdown_table, format_latex_snippet

smoke_summary = {
    "metadata": {
        "model_base": MODEL_ID,
        "eval_set": "eval_unseen_apis.jsonl (smoke slice)",
        "eval_count": SMOKE_EVAL_SIZE,
        "mode": "rapid_smoke_test",
    },
    "configs": {
        "base_model": base_results,
        "sft_only": sft_results,
        "sft_dpo_vericall": dpo_results,
    },
    "mcnemar_tests": {
        "vericall_vs_base": {"statistic": stat_base, "p_value": p_base},
        "vericall_vs_sft": {"statistic": stat_sft, "p_value": p_sft},
    }
}

Path("results").mkdir(exist_ok=True)
with open("results/eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(smoke_summary, f, indent=2)

print("[✓] Saved smoke test results to results/eval_summary.json")
!python scripts/render_results.py
